## LETTURA EQUIPMENT MIDAS CUPID NMV 
- lettera logbook google
- lettura equipment digitalizzatore (DIG0, DGH0)
- lettura equipment oscilloscopio (HS00, DS00, HS01, DS01, HS02, DS02, HS03, DS03)
## LOGBOOK:

In [ ]:
import site
print(site.ENABLE_USER_SITE)
print(site.getusersitepackages())

In [ ]:
import wclib as wc
# lettura logbook
logbook =  wc.panda_from_gspreadsheet(key='158Nn7Y59soXvZsv7gih2xP1vzWCiMpzk4hIN6LMxDok', sheet_name='log')
print()
print(">> description:", logbook[(logbook.run==178)].description.values[0])
print(">> table and value:")
logbook.tail(1)

## DIGITALIZZATORE:

In [ ]:
%matplotlib inline
# codice per la lettira eventi equipment digitalizattore v7014
import numpy as np
import matplotlib.pyplot as plt
import os
import midas.file_reader
from datetime import datetime
import wclib as wc
import warnings

# logbook https://docs.google.com/spreadsheets/d/158Nn7Y59soXvZsv7gih2xP1vzWCiMpzk4hIN6LMxDok/
logbook =  wc.panda_from_gspreadsheet(key='158Nn7Y59soXvZsv7gih2xP1vzWCiMpzk4hIN6LMxDok', sheet_name='log')

warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")

def Gauss3(x, a0, x0, s0):
    import numpy as np
    return a0 * np.exp(-(x - x0)**2 / (2 * s0**2))

def fittalo(func, x, y, ax, p0, fmt='k'):
  from scipy.optimize import curve_fit
  from sklearn.metrics import r2_score
  popt, pcov = curve_fit(func,x, y, p0=p0)
  perr = np.sqrt(np.diag(pcov))
  r2=r2_score(y, func(x, *popt))
  xf = np.linspace(x.min(), x.max(), 100)
  ax.plot(xf, func(xf, *popt), fmt, label='a = {0:.2e}±{1:.2e}\nb = {2:.2e}±{3:.2e} \
  \nc = {4:.2e}±{5:.2e}\nr^2 = {6:.3f}'.format(popt[0], perr[0], popt[1], perr[1], popt[2], perr[2], r2))
  return ax, popt

def plot_waveform(waveform, lenw, pmt, event_number, event_time):
    import numpy as np
    
    t = np.linspace(0,lenw, lenw)
    for ipmt in range(pmt):
        plt.subplot(pmt, 1, ipmt+1)
        plt.plot(t, waveform[ipmt])

    return

def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

verbose = False
DEBUG   = False
run     = int(input("run number [int] "))

outplot = True # fa i plot

path    = '/home/mazzitel/nmv-data/WC/WC25'

    
mfile = wc.open_mid(run=run, path=path, cloud=False, tag='', verbose=verbose)

# esempio lettura informazioni dall'odb  #######
odb = wc.get_bor_odb(mfile)

try:
    Run_description   = odb.data['Experiment']['Run Parameters']['Run description']
    print('Run description:', Run_description)
    # uso gli aggiornamenti del logbook:
    Run_description=logbook[(logbook.run==run)].description.values[0]
    MOD=Run_description[4:5]
    print("module:", MOD)
    match int(MOD):
        case 0:
            p1=0
            p2=1
        case 1:
            p1=2
            p2=3
        case 2:
            p1=4
            p2=5
except:
    print('WARNING: no run description')

DigitizerSamples  = odb.data['Configurations']['DigitizerSamples']
DigitizerPostTrg  = odb.data['Configurations']['DigitizerPostTrg']
print('DigitizerSamples:', DigitizerSamples)
print('DigitizerPostTrg:', DigitizerPostTrg)
corrected = odb.data['Configurations']['DRS4Correction']
channels_offset = odb.data['Configurations']['DigitizerOffset']
###############################################
# lettura equipment nel file #######

waveform1=[]
waveform2=[]
waveform1p=[]
waveform2p=[]
waveform1t=[]
waveform2t=[]


calo_waveform=[]
calo_waveform_all=[]
t0s=[]
offset = 90
fig, ax=plt.subplots(3,3,figsize=(10,8))


for event in mfile:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        # print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        # print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    xs   = 200 #nuovo riferimento
    x1wc = 290
    wwc  = 200
    x2wc = x1wc+wwc
    for bank_name, bank in event.banks.items():
        if ('DGH0' in bank_name): # PMTs wavform 
            waveform_header = wc.daq_dgz_full2header(bank, verbose=False)
            if verbose: print (waveform_header)
            waveform = wc.daq_dgz_full2array(event.banks['DIG0'], waveform_header, verbose=False, ch_offset=channels_offset)
            
            calo_base = int(np.mean(waveform[7][:xs]))
            wc1_base  = int(np.mean(waveform[p1][:xs]))
            wc2_base  = int(np.mean(waveform[p2][:xs]))
            if verbose or event_number % 1000==0:
                print("baseliane", calo_base, wc1_base, np.std(waveform[p1][:xs]), wc2_base, np.std(waveform[p2][:xs]))
            
            trig = np.array(waveform[6])
            calo = np.array(waveform[7])-calo_base
            wc1  = np.array(waveform[p1])-wc1_base
            wc2  = np.array(waveform[p2])-wc2_base
            
            der=np.diff(trig)
            t0=np.argmin(der)
            t0s.append(t0)
            
            dt=t0-xs
            trig_norm = trig[dt:][:1000] 
            calo_norm = calo[dt:][:1000]
            wc1_norm = wc1[dt:][:1000]
            wc2_norm = wc2[dt:][:1000]
            
            # ch 8 trigger, ch1 Tr up, ch2 Tr down, ch 3 right, ch 4 left (from PMP PS)
            waveform1.append(np.trapezoid(wc1_norm[x1wc:x2wc]))
            waveform2.append(np.trapezoid(wc2_norm[x1wc:x2wc]))
            waveform1p.append(np.min(wc1_norm[x1wc:x2wc]))
            waveform2p.append(np.min(wc2_norm[x1wc:x2wc]))
            waveform1t.append(np.argmin(wc1_norm[x1wc:x2wc]))
            waveform2t.append(np.argmin(wc2_norm[x1wc:x2wc]))


            calo_waveform.append(np.trapezoid(calo_norm[300:400]))
            calo_waveform_all.append(np.trapezoid(calo))
            if event_number % 20==0:
                ax[0,0].plot(np.linspace(t0-10, t0+10, 20), trig[t0-10:t0+10])
                ax[0,2].plot(np.linspace(190, 210, 20), trig_norm[190:210])

                ax[1,0].plot(np.linspace(300, 400, 100), calo_norm[300:400])
                ax[1,1].plot(np.linspace(x1wc, x2wc, wwc), wc1_norm[x1wc:x2wc])
                ax[1,2].plot(np.linspace(x1wc, x2wc, wwc), wc2_norm[x1wc:x2wc])
                
            if DEBUG:
                plt.show()
    if DEBUG:        
        if event.header.serial_number == 20: # si ferm dopo i primi 20 eventi
            break
            
calo_waveform = np.array(calo_waveform)
waveform1     = np.array(waveform1)
waveform2     = np.array(waveform2)
waveform1p    = np.array(waveform1p)
waveform2p    = np.array(waveform2p)
dtsample      = odb.data['Configurations']['DigitizerSamples']/(odb.data['Configurations']['SamplingFrequency']*1e6)
waveform1t    = np.array(waveform1t)*dtsample
waveform2t    = np.array(waveform2t)*dtsample



x1calo = -7000
x2calo = -2000
bbins  = 100
xm1    = -2000
xm2    = 1000

ax[0,1].hist(t0s, bins=bbins)
ax[2,0].hist(calo_waveform, bins=bbins, range=(np.min(calo_waveform), np.max(calo_waveform)))
ax[2,1].hist(waveform1, bins=bbins, range=(xm1, xm2))
ax[2,2].hist(waveform2, bins=bbins, range=(xm1, xm2))

cc,_ =np.histogram(calo_waveform[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(np.min(calo_waveform), np.max(calo_waveform)))
cw1,_=np.histogram(waveform1[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(xm1, xm2)) 
cw2,_=np.histogram(waveform2[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(xm1, xm2))

ax[2,0].hist(calo_waveform[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(np.min(calo_waveform), np.max(calo_waveform)))
ax[2,1].hist(waveform1[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(xm1, xm2))
ax[2,2].hist(waveform2[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(xm1, xm2))


plt.show()

fig, axx=plt.subplots(2,2,figsize=(10,8))

cutcw1p=(calo_waveform>x1calo) & (calo_waveform<x2calo) & (waveform1p<-5)
cutcw2p=(calo_waveform>x1calo) & (calo_waveform<x2calo) & (waveform2p<-13)
cutt = cutcw1p & cutcw2p



cw1p,_=np.histogram(waveform1p[cutcw1p], bins=bbins, 
             range=(-700, 20)) 
cw2p,_=np.histogram(waveform2p[cutcw2p], bins=bbins, 
             range=(-700, 20))
axx[0,0].hist(waveform1p[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(-700, 20))
axx[0,1].hist(waveform2p[(calo_waveform>x1calo) & (calo_waveform<x2calo)], bins=bbins, 
             range=(-700, 20))
axx[0,0].hist(waveform1p[cutcw1p], bins=bbins, 
             range=(-700, 20))
axx[0,1].hist(waveform2p[cutcw2p], bins=bbins, 
             range=(-700, 20))

axx[1,0].hist(waveform1t[cutt]+waveform2t[cutt], bins=bbins)
axx[1,1].hist(waveform1t[cutt]-waveform2t[cutt], bins=bbins)
y,x=np.histogram(waveform1t[cutt]-waveform2t[cutt], bins=bbins)

axx[1,1], par = fittalo(Gauss3,x[:-1],y, axx[1,1], [y.max(),0, 1e-5], fmt='k--')
plt.legend()
#axx[1,0].plot(waveform1p[cutcw1p], waveform1[cutcw1p], 'k.')
#axx[1,1].plot(waveform2p[cutcw2p], waveform2[cutcw2p], 'r.')
#axx[1,0].set_xlabel('peak')
#axx[1,1].set_xlabel('peak')
#axx[1,0].set_ylabel('charge')
#axx[1,1].set_ylabel('cherge')


plt.show()

print("calo: ", np.sum(cc))
print("wc1 int noTh: ",  np.sum(cw1), np.sum(cw1)/np.sum(cc))
print("wc2 int noTh: ",  np.sum(cw2), np.sum(cw2)/np.sum(cc))
print("wc1 min Th: ",  np.sum(cw1p), np.sum(cw1p)/np.sum(cc))
print("wc2 min Th: ",  np.sum(cw2p), np.sum(cw2p)/np.sum(cc))
print("coinc: ",  np.sum(cutt), np.sum(cutt)/np.sum(cc))
print("tsum", np.mean(waveform1t[cutt]+waveform2t[cutt]), np.std(waveform1t[cutt]+waveform2t[cutt]))
print("Dt", np.mean(waveform1t[cutt]-waveform2t[cutt]), np.std(waveform1t[cutt]-waveform2t[cutt]))


print('DONE')


In [ ]:
calocat=1.0115e6
caloune=1.006e6
calo_waveform=np.array(calo_waveform)
plt.hist(calo_waveform, bins=100, range=(4.13e6, 4.17e6))
plt.hist(calo_waveform[(calo_waveform>caloune) & (calo_waveform<calocat)], bins=100, range=(4.13e6, 4.17e6))
plt.show()

In [ ]:
plt.hist(calo_waveform_all, bins=100)
plt.show()

## events display:

In [ ]:
# show events one by one
from IPython import display
run     = int(input("run number [int] "))
path    = '/home/mazzitel/nmv-data/WC/WC25'
mfile = wc.open_mid(run=run, path=path, cloud=False, tag='', verbose=verbose)
DigitizerSamples  = odb.data['Configurations']['DigitizerSamples']
DigitizerPostTrg  = odb.data['Configurations']['DigitizerPostTrg']
print('DigitizerSamples:', DigitizerSamples)
print('DigitizerPostTrg:', DigitizerPostTrg)
corrected = odb.data['Configurations']['DRS4Correction']
channels_offset = odb.data['Configurations']['DigitizerOffset']

for event in mfile:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    for bank_name, bank in event.banks.items():
        if ('DGH0' in bank_name): # PMTs wavform 
            waveform_header = wc.daq_dgz_full2header(bank, verbose=False)
            if verbose: print (waveform_header)
            waveform = wc.daq_dgz_full2array(event.banks['DIG0'], waveform_header, verbose=False, ch_offset=channels_offset)

            fig, ax = plt.subplots(8,4,figsize=(10,10))
            pmts=32
            y = 0
            x = 0
            for pmt in range(0,pmts):
                x = int(pmt/4)
                y += 1
                if pmt % 4 == 0:
                     y=0 
                ax[x,y].plot(np.linspace(0,1024,1024), waveform[pmt])    
    next=input("press return to next or type exit")
    if next=='exit':
        break
    display.clear_output(wait=True)
    display.display(plt.gcf())

## cerca spike (problema caen) e fa i plot o singoli o sovapporti:

In [ ]:
# cerca spike

import numpy as np
import matplotlib.pyplot as plt
import midas.file_reader
from datetime import datetime
import wclib as wc

from scipy.ndimage import maximum_filter1d

# ----------------------    
path    = '/home/mazzitel/nmv-data/WC/WC25'
pmts    = 16
spmt    = 0
w_mean  = np.zeros(pmts)
w_std   = np.zeros(pmts)
w_th    = np.zeros(pmts)
w_m     = np.zeros((pmts, 1024))
window  = 3
mw      = np.zeros((pmts, 1024), dtype=bool)
xs      = 20
n_coincidence=0
verbose = False
# -------------------------
logbook     =  wc.panda_from_gspreadsheet(key='158Nn7Y59soXvZsv7gih2xP1vzWCiMpzk4hIN6LMxDok', sheet_name='log')

run         = int(input("run number [int] "))
enable_plot = input("Plots? [y/n] ").strip().lower()

if enable_plot in ("y", "yes"):
    plots = True
    enable_persistence = input("persistence/sigle? [y/n] ").strip().lower()
else:
    plots = False
    
if enable_persistence in ("y", "yes"):
    persistence = True
    fig, ax = plt.subplots(int(pmts/2),2,figsize=(10,5))
else:
    persistence = False

mfile   = wc.open_mid(run=run, path=path, cloud=False, tag='', verbose=verbose)
# ----------------------------
odb = wc.get_bor_odb(mfile)

try:
    Run_description   = odb.data['Experiment']['Run Parameters']['Run description']
    print('Run description:', Run_description)
    # uso gli aggiornamenti del logbook:
    Run_description=logbook[(logbook.run==run)].description.values[0]
    print('Run description in logbook:', Run_description)
except:
    print('WARNING: no run description')

DigitizerSamples  = odb.data['Configurations']['DigitizerSamples']
DigitizerPostTrg  = odb.data['Configurations']['DigitizerPostTrg']
print('DigitizerSamples:', DigitizerSamples)
print('DigitizerPostTrg:', DigitizerPostTrg)
corrected = odb.data['Configurations']['DRS4Correction']
channels_offset = odb.data['Configurations']['DigitizerOffset']

for event in mfile:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    for bank_name, bank in event.banks.items():
        if ('DGH0' in bank_name): # PMTs wavform 
            waveform_header = wc.daq_dgz_full2header(bank, verbose=False)
            if verbose: print (waveform_header)
            waveform = wc.daq_dgz_full2array(event.banks['DIG0'], waveform_header, verbose=False, ch_offset=channels_offset)
            trig = np.array(waveform[6])
            
            der=np.diff(trig)
            t0=np.argmin(der)
            dt=t0-xs
            for pmt in range(0,pmts):
                #waveform[pmt]=waveform[pmt][dt:][:1000]
                w_mean[pmt] = np.mean(waveform[pmt][0:100])
                w_std[pmt]  = np.std(waveform[pmt][0:100])
                w_th[pmt]   = (w_mean[pmt] - 5 * w_std[pmt])
                w_m[pmt]    = waveform[pmt] < w_th[pmt]                      # bool
                mw[pmt]     = maximum_filter1d(w_m[pmt], size=2*window+1)    # restituisce bool
                # if verbose: 
                #     print(pmt, w_mean[pmt], w_std[pmt], w_th[pmt])
            coincidence = (mw[spmt] & mw[spmt+1] & mw[spmt+2] & mw[spmt+3] & mw[spmt+4] & mw[spmt+5])
            indices = np.where(coincidence)[0]
            if len(indices):
                if plots:
                    #print([c for c in mw], coincidence, indices)
                    if not persistence:
                        fig, ax = plt.subplots(int(pmts/2),2,figsize=(10,5))
                    y = 0
                    x = 0
                    for pmt in range(0,pmts):
                        x = int(pmt/2)
                        y += 1
                        if pmt % 2 == 0:
                             y=0 
                        ax[x,y].plot(np.linspace(0,1000,1000), waveform[pmt][:1000])
                        ax[x,y].hlines(y=w_th[pmt], xmin=0, xmax=1000, colors='g', linestyles=':')
                    if not persistence:
                        plt.show()
                n_coincidence+=1
if persistence:
    plt.show()
print(n_coincidence)
print("DONE")

## LETTURA EQUIPMENT OSCILLOSCOPIO

In [ ]:
# creazione del DB
runsvalue = {}


In [ ]:
# === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import midas.file_reader
from datetime import datetime
import wclib as wc
import json

def plot_waveform(waveform, lenw, pmt, event_number, event_time):
    import numpy as np
    
    t = np.linspace(0,lenw, lenw)
    for ipmt in range(pmt):
        plt.subplot(pmt, 1, ipmt+1)
        plt.plot(t, waveform[ipmt])
        plt.ylabel('ch{:d} [mV]'.format(ipmt))

    return

def plot_waveform_h(waveform, lenw, pmt, hmin, hmax, vmin, vmax, event_number, event_time):
    import numpy as np
    import math
    npx = math.ceil(pmt/2)
    t = np.linspace(0,lenw, lenw)
    for ipmt in range(pmt):
        plt.subplot(npx, 2, ipmt+1)
        plt.plot(t[hmin:hmax], waveform[ipmt, ][hmin:hmax])
        if vmax!=4096 or vmin!=0: 
            plt.ylim(vmin, vmax)
        plt.ylabel('ch{:d} [mV]'.format(ipmt))

    return


verbose = False
DEBUG   = False
PLOT    = False
run     = int(input("run number [int] "))
pmt_n   = 4
hmin = 0
hmax = 1024
vmin = 0
vmax =4096

outplot = True # fa i plot

path    = '/home/mazzitel/nmv-data/WC/WC25'

mfile = wc.open_mid(run=run, path=path, cloud=False, tag='', verbose=verbose)

# esempio lettura informazioni dall'odb  #######
odb = wc.get_bor_odb(mfile)

try:
    Run_description   = odb.data['Experiment']['Run Parameters']['Run description']
    print('Run description:', Run_description)
except:
    print('WARNING: no run description')

###############################################
# lettura equipment nel file #######

runsvalue[run] = {
    "tsvalue": [],   # comune a tutte le waveform
    "tsunix": [],    # comune a tutte le waveform
    "charge": {      # per waveform/canale
        0: [],       # ch1
        1: [],       # ch2
        2: [],       # ch3
        3: []        # ch4
    }
}
for event in mfile:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("%s, banks %s" % (datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))
    for bank_name, bank in event.banks.items():
        runsvalue[run]["tsunix"].append(event.header.timestamp)
        # if ('DGH0' in bank_name): # PMTs wavform 

        
        if bank_name=='HS00': 
            runsvalue[run]["tsvalue"].append(hd['TRIGGER_TIME'])
            hd = json.loads(bytes(event.banks['HS00'].data).decode("utf-8"))
            tw = np.asarray(event.banks['DS00'].data, dtype=np.float64)
            lenw=int(hd['WAVE_ARRAY_COUNT'])
            # init new waveform
            waveform=np.zeros((pmt_n,lenw), dtype=np.float64)
            if len(tw)>0:
                waveform[0]=tw
                runsvalue[run]["charge"][0].append(np.trapezoid(tw))
        if bank_name=='HS01': 
            hd = json.loads(bytes(event.banks['HS01'].data).decode("utf-8"))
            tw = np.asarray(event.banks['DS01'].data, dtype=np.float64)
            lenw=int(hd['WAVE_ARRAY_COUNT'])
            if len(tw)>0:
                waveform[1]=tw
                runsvalue[run]["charge"][1].append(np.trapezoid(tw))
        if bank_name=='HS02': 			
            hd = json.loads(bytes(event.banks['HS02'].data).decode("utf-8"))
            tw = np.asarray(event.banks['DS02'].data, dtype=np.float64)
            lenw=int(hd['WAVE_ARRAY_COUNT'])
            if len(tw)>0:
                waveform[2]=tw
                runsvalue[run]["charge"][2].append(np.trapezoid(tw))
        if bank_name=='HS03': 
            hd = json.loads(bytes(event.banks['HS03'].data).decode("utf-8"))
            tw = np.asarray(event.banks['DS03'].data, dtype=np.float64)
            lenw=int(hd['WAVE_ARRAY_COUNT'])
            if len(tw)>0:
                waveform[3]=tw
                runsvalue[run]["charge"][3].append(np.trapezoid(tw))
    if PLOT: 
        hmax=lenw   
        fig = plt.figure(figsize=(9,4), facecolor='#DEDEDE')
        plot_waveform_h(waveform, lenw, pmt_n, hmin, hmax, vmin, vmax, event_number, event_time)
        plt.show()
            
    if DEBUG:        
        if event.header.serial_number == 20: # si ferm dopo i primi 20 eventi
            break

from datetime import datetime, timezone

# trigger_times: lista di stringhe ISO 8601, una per evento
# es: "2026-01-16T17:21:25.123456"
def parse_iso(ts: str) -> float:
    # Supporta anche senza frazioni; assume timezone locale se assente (meglio: usa UTC se puoi)
    dt = datetime.fromisoformat(ts)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp()

t = np.array([parse_iso(s) for s in runsvalue[run]["tsvalue"]], dtype=float)
t.sort()

dt = np.diff(t)  # secondi tra trigger

Tmean = dt.mean()
f_avg = 1.0 / Tmean

jitter_rms = dt.std(ddof=1)          # secondi
jitter_rel = jitter_rms / Tmean      # frazione

print("Periodo medio [s]:", Tmean)
print("Frequenza media [Hz]:", f_avg)
print("Jitter RMS [s]:", jitter_rms)
print("Jitter relativo:", jitter_rel)

N = len(runsvalue[run]["tsunix"])
T = runsvalue[run]["tsunix"][-1] - runsvalue[run]["tsunix"][0]

f_avg = (N - 1) / T if T > 0 else float("nan")

print("Frequenza media [Hz]:", f_avg)

print('V/div: ~', 10*hd['VERTICAL_GAIN']*(hd['MAX_VALUE']-hd['MIN_VALUE'])/8)
print('DONE')

In [ ]:
plt.hist(np.array(runsvalue[run]["charge"][0]), 100, density=True, range=(0,100), label='run: {}'.format(run))
plt.legend()
plt.show()

In [ ]:
plt.hist(np.array(runsvalue[222]["charge"][0]), 100, density=True, range=(0,100), label='run: {}'.format(222))
plt.hist(np.array(runsvalue[223]["charge"][0]), 100, density=True, range=(0,100), label='run: {}'.format(223), alpha=0.7)
plt.legend()
plt.show()


In [ ]:
t


In [ ]:
dt = np.diff(t)

In [ ]:
dt

In [ ]:
np.size(t)

In [ ]:
n = 3182166080

XX = (n >> 8) & 0xFF
Y  = (n >> 4) & 0xF
Z  = n & 0xF

print("caret:",hex(XX), "scheda:",hex(Y), "canale:",hex(Z))